### SQL & PySpark basics Review 

Objectives:

Review the most common operations in SQL and PySpark : dates, filters, aggregations, and calculated columns; distinguish between transformations, actions, and lazy evaluation.

In [0]:
from pyspark.sql import functions as F

In [0]:
df_transaction = spark.read.table("samples.bakehouse.sales_transactions")

In [0]:
%sql
DESCRIBE TABLE samples.bakehouse.sales_transactions
-- show the column information 

#### 1.1 - Exploration et initial quality 

##### Display 20 rows without using SELECT *;

In [0]:
%sql 
SELECT transactionID, datetime, product, quantity, unitPrice, totalPrice
FROM samples.bakehouse.sales_transactions
LIMIT 20 

In [0]:
df_transaction.select("transactionID", "dateTime", "product", "quantity", "unitPrice","totalPrice").show(20, truncate=False)

##### Count the number of transactions and distinct customers;

In [0]:
%sql 
SELECT COUNT(transactionID) AS total_transactions, COUNT(DISTINCT customerID) as total_customers
FROM samples.bakehouse.sales_transactions

In [0]:
df_transaction.agg(
    F.count("transactionID").alias("total_transaction"),
    F.countDistinct("customerID").alias("total_customer")
).show()

##### Get the minimum and maximum dates;

In [0]:
%sql
SELECT MIN(DATE(dateTime)) as min_date, MAX(DATE(dateTime)) as max_date
FROM samples.bakehouse.sales_transactions 

In [0]:
df_transaction.agg(
    F.min(F.to_date("dateTime")).alias("min_date"),
    F.max(F.to_date("dateTime")).alias("max_date")
).show()

##### Calculate the minimum, maximum, average, and approximate median of quantity, unitPrice, and totalPrice

In [0]:
%sql
SELECT
    'min' AS stat,
    MIN(quantity) AS quantity,
    MIN(unitPrice) AS unitPrice,
    MIN(totalPrice) AS totalPrice
FROM samples.bakehouse.sales_transactions

UNION ALL

SELECT
    'max',
    MAX(quantity),
    MAX(unitPrice),
    MAX(totalPrice)
FROM samples.bakehouse.sales_transactions

UNION ALL

SELECT
    'avg',
    ROUND(AVG(quantity), 2),
    ROUND(AVG(unitPrice), 2),
    ROUND(AVG(totalPrice), 2)
FROM samples.bakehouse.sales_transactions

UNION ALL

SELECT
    'median',
    MEDIAN(quantity),
    MEDIAN(unitPrice),
    MEDIAN(totalPrice)
FROM samples.bakehouse.sales_transactions ; 

In [0]:
def stat(col_name):
    return [
        F.min(col_name).alias(f"min_{col_name}"),
        F.max(col_name).alias(f"max_{col_name}"),
        F.round(F.avg(col_name), 2).alias(f"avg_{col_name}"),
        F.median(col_name).alias(f"median_{col_name}")]

df_stats = df_transaction.select(
    *stat("quantity"),
    *stat("unitPrice"),
    *stat("totalPrice")
)

df_unpivot = df_stats.unpivot(
    ids=[],
    values=df_stats.columns,
    variableColumnName="variable",
    valueColumnName="value"
)

df_split = (
    df_unpivot.withColumn("stat", F.split("variable", "_")[0])
    .withColumn("variable", F.split("variable", "_")[1])
    )

df_split.groupBy("stat").pivot("variable").agg(F.first("value")).display()

##### Count null values for each column

In [0]:
%sql
SELECT 
    SUM(CASE WHEN transactionID IS NULL THEN 1 ELSE 0 END) AS null_transactions,
    SUM(CASE WHEN customerID IS NULL THEN 1 ELSE 0 END) AS null_customers,
    SUM(CASE WHEN franchiseID IS NULL THEN 1 ELSE 0 END) AS null_franchise,
    SUM(CASE WHEN dateTime IS NULL THEN 1 ELSE 0 END) AS null_date,
    SUM(CASE WHEN product IS NULL THEN 1 ELSE 0 END) AS null_product,
    SUM(CASE WHEN quantity IS NULL THEN 1 ELSE 0 END) AS null_quantity,
    SUM(CASE WHEN unitPrice IS NULL THEN 1 ELSE 0 END) AS null_unitPrice,
    SUM(CASE WHEN totalPrice IS NULL THEN 1 ELSE 0 END) AS null_totalPrice,
    SUM(CASE WHEN paymentMethod IS NULL THEN 1 ELSE 0 END) AS null_paymentMethod,
    SUM(CASE WHEN cardNumber IS NULL THEN 1 ELSE 0 END) AS null_cardNumber
FROM samples.bakehouse.sales_transactions

In [0]:
df_transaction.select(
    [F.count(
        F.when(F.col(col).isNull(), 1)
    ).alias(f"null_{col}") for col in df_transaction.columns]
).show()

##### Check for negative prices, zero or negative quantities, and invalid dates;



In [0]:
%sql
SELECT transactionID, quantity, unitPrice, totalPrice, dateTime
FROM samples.bakehouse.sales_transactions
WHERE 
    quantity <= 0 OR 
    unitPrice <= 0 OR 
    totalPrice <= 0 OR 
    TRY_CAST(dateTime AS timestamp) IS NULL

In [0]:
df_transaction.filter(
    (F.col("quantity") <= 0) |
    (F.col("unitPrice") <= 0) |
    (F.col("totalPrice") <= 0) |
    F.try_to_timestamp("dateTime").isNull()
).show()

##### Verify whether totalPrice is consistent with quantity * unitPrice.

In [0]:
%sql
SELECT *
FROM samples.bakehouse.sales_transactions
WHERE quantity*unitPrice <> totalPrice

In [0]:
df_transaction.filter(
    F.col("totalPrice") != F.col("quantity") * F.col("unitPrice")
).show()

#### 1.2 - Time and Product Aggregations

##### Revenue and sales volume by day

In [0]:
%sql
SELECT DATE(dateTime), SUM(totalPrice) AS total_price_day, SUM(quantity) AS total_quantity_day
FROM samples.bakehouse.sales_transactions
GROUP BY DATE(dateTime)
ORDER BY dateTime ASC

In [0]:
df_transaction.groupBy(
    F.to_date("dateTime").alias("dateTime")
    ).agg(
        F.sum("quantity").alias("total_quantity_day"),
        F.sum("totalPrice").alias("total_price_day")
    ).show()

##### Revenue and sales volume by month

In [0]:
%sql
SELECT MONTH(dateTime) as month, SUM(totalPrice) as total_price_by_month, SUM(quantity) as total_quantity_by_month
FROM samples.bakehouse.sales_transactions
GROUP BY MONTH(dateTime)

In [0]:
df_transaction.groupBy(
    F.month("dateTime").alias("month")
).agg(
    F.sum("quantity").alias("total_quantity_month"),
    F.sum("totalPrice").alias("total_price_month")
).show()

##### Average transaction value

In [0]:
%sql
SELECT ROUND(AVG(totalPrice),2) as avg_transa_value
FROM samples.bakehouse.sales_transactions

In [0]:
df_transaction.select(
    F.round(F.avg("totalPrice"),2).alias("avg_totalPrice")
).show()

##### Best-selling product by quantity

In [0]:
%sql
SELECT product, SUM(quantity) as total_quantity
FROM samples.bakehouse.sales_transactions
GROUP BY product
ORDER BY total_quantity DESC
LIMIT 1




In [0]:
df_transaction.groupBy(
    F.col("product")
    ).agg(
         F.sum("quantity").alias("total_quantity_product")
        ).orderBy(
            F.desc("total_quantity_product")
        ).show(1)

##### Product having the highest revenue

In [0]:
%sql
SELECT product, SUM(totalPrice) as total_price
FROM samples.bakehouse.sales_transactions
GROUP BY product
ORDER BY total_price DESC
LIMIT 1

In [0]:
df_transaction.groupby(
    F.col("product")
).agg(
    F.sum("totalPrice").alias("total_price_product")
).orderBy(F.desc("total_price_product")).show(1)

##### Divide by payment method

In [0]:
%sql 
SELECT paymentMethod, COUNT(*) as total
FROM samples.bakehouse.sales_transactions
GROUP BY paymentMethod
ORDER BY total DESC


In [0]:
result = df_transaction.groupBy(
    F.col("paymentMethod")
).agg(
    F.count("transactionID").alias("total_transactions_paymentMethod")
).orderBy(F.desc("total_transactions_paymentMethod"))

result.display()


##### Busiest hour and day of the week

In [0]:
%sql
SELECT HOUR(dateTime) AS hour, COUNT(*) AS total
FROM samples.bakehouse.sales_transactions
GROUP BY hour
ORDER BY total DESC

In [0]:
%sql
SELECT DAYOFWEEK(dateTime) AS dayOfWeek, COUNT(*) AS total
FROM samples.bakehouse.sales_transactions
GROUP BY dayOfWeek
ORDER BY total DESC

In [0]:
df_transaction.groupBy(
    F.hour("dateTime").alias("hour")
).agg(
    F.count("transactionID").alias("total_transactions_hour")
).orderBy(F.desc("total_transactions_hour")).display()

In [0]:
df_transaction.groupBy(
    F.dayofweek("dateTime").alias("dayOfWeek")
).agg(
    F.count("transactionID").alias("total_transactions_hour")
).orderBy(F.desc("total_transactions_hour")).display()